# BranchCache: free-tier GPU serving benchmark

Runs the naive / vLLM / SGLang serving comparison from `eval/run_serving_benchmark.py` against a real model. No paid services anywhere, this only needs the free GPU tier and no billing info.

Default model: `Qwen/Qwen2.5-Coder-3B-Instruct` (set `BRANCHCACHE_MODEL` to override, e.g. drop to `Qwen/Qwen2.5-Coder-1.5B-Instruct` if you OOM at high N).


**Kaggle-specific:** Notebook settings → Accelerator → GPU T4 x2 (only one GPU gets used, T4 x2 is just what Kaggle offers) and Internet → On, needed for pip install and the git clone. Free quota is a 9-hour session limit and 30 GPU-hours a week. If the sweep doesn't finish in one session, start a fresh one and rerun the notebook, completed rows in the CSV get skipped automatically.

## 1. Clone the repo and install

In [ ]:
import os
import sys

REPO_URL = "https://github.com/OmkarKashyap/BranchCache.git"
if not os.path.exists("BranchCache"):
    !git clone $REPO_URL
%cd BranchCache
!pip install -q -e ".[dev]"


`vllm` and `sglang` get installed separately from here, each in its own cell with `-v`. They're the heaviest and most fragile installs here, pulling in native CUDA sub-packages that sometimes fail to build. Splitting them means a failure in one doesn't block the other or the tests above.

In [ ]:
!pip install -v vllm 2>&1 | tail -80

In [ ]:
!pip install -v sglang 2>&1 | tail -80

`vllm serve` can fail to even start with an `ImportError` on `openai.types.responses`. Its CLI needs a newer `openai` SDK than what's on the system. Our own client code only uses the stable `chat.completions.create` call, so upgrading here is safe.

In [ ]:
!pip install -q -U openai
!python -c "from openai.types.responses import NamespaceTool; print('openai SDK OK')"

In [ ]:
!nvidia-smi

Sanity check: the CPU-only unit tests should still pass here, same as on your laptop.

In [ ]:
!{sys.executable} -m pytest tests/ -q

## 2. Helpers

One server at a time keeps memory pressure predictable on a single free GPU. We bring a server up, run its slice of the sweep, tear it down, then move on. `run_serving_benchmark.py` checkpoints to CSV as it goes, so nothing gets redone if a cell is rerun.

In [ ]:
import os
import subprocess
import time

import requests


def start_server(cmd, log_path):
    log = open(log_path, "w")
    return subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)

def wait_for_server(port, log_path, timeout_s=600):
    url = f"http://localhost:{port}/health"
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            if requests.get(url, timeout=2).status_code == 200:
                print(f"server on port {port} is up")
                return
        except Exception:
            pass
        time.sleep(5)
    raise RuntimeError(f"server on port {port} never came up, check {log_path}")

def stop_server(proc):
    proc.terminate()
    try:
        proc.wait(timeout=15)
    except subprocess.TimeoutExpired:
        proc.kill()


## 3. Naive baseline (prefix caching disabled)

Same vLLM server as the caching run below, just started with `--no-enable-prefix-caching`. That isolates caching as the one thing changing between the two runs.

In [ ]:
%env BRANCHCACHE_MODEL=Qwen/Qwen2.5-Coder-3B-Instruct
%env BRANCHCACHE_NAIVE_URL=http://localhost:8000/v1

naive_proc = start_server(
    ['vllm', 'serve', os.environ['BRANCHCACHE_MODEL'], '--port', '8000',
     '--no-enable-prefix-caching', '--gpu-memory-utilization', '0.85',
     '--max-model-len', '8192'],
    'naive.log',
)
wait_for_server(8000, 'naive.log')


In [ ]:
!{sys.executable} -m eval.run_serving_benchmark --strategies naive --ns 1 2 4 8 --trials 3

In [ ]:
stop_server(naive_proc)

## 4. vLLM automatic prefix caching

In [ ]:
%env BRANCHCACHE_VLLM_URL=http://localhost:8001/v1

vllm_proc = start_server(
    ['vllm', 'serve', os.environ['BRANCHCACHE_MODEL'], '--port', '8001',
     '--enable-prefix-caching', '--gpu-memory-utilization', '0.85',
     '--max-model-len', '8192'],
    'vllm.log',
)
wait_for_server(8001, 'vllm.log')


In [ ]:
!{sys.executable} -m eval.run_serving_benchmark --strategies vllm --ns 1 2 4 8 --trials 3

In [ ]:
stop_server(vllm_proc)

## 5. SGLang RadixAttention

RadixAttention is on by default in SGLang, no flag needed.

`sglang` pins an exact `openai==2.6.1`, older than what `vllm` needed above. `sglang.launch_server` fails to start against the newer version, the same way `vllm serve` failed against the older one, so we pin it back down right before this server starts.

In [ ]:
!pip install -q "openai==2.6.1"

In [ ]:
%env BRANCHCACHE_SGLANG_URL=http://localhost:8002/v1

sglang_proc = start_server(
    [sys.executable, '-m', 'sglang.launch_server', '--model-path', os.environ['BRANCHCACHE_MODEL'],
     '--port', '8002'],
    'sglang.log',
)
wait_for_server(8002, 'sglang.log')


In [ ]:
!{sys.executable} -m eval.run_serving_benchmark --strategies sglang --ns 1 2 4 8 --trials 3

In [ ]:
stop_server(sglang_proc)

## 6. Done

`results/raw/serving_benchmark.csv` now has rows for all three strategies. Pull it down, or commit it from here if this environment has your git credentials.

If the session dies partway through, just rerun the notebook top to bottom. Completed rows get skipped automatically.